# W2 · Vorlesung · Zufall und Verteilung

Dieses Notebook zeige ich in der Vorlesung live. Schätzen Sie vor jeder Zelle, was herauskommt. Nach der Vorlesung
können Sie alles selbst ausführen und jede Zahl der Folien nachrechnen (Teil 6).

**Fester Seed:** `np.random.default_rng(2026)` liefert bei jedem Lauf dieselben Zufallszahlen. Deshalb sehen Sie hier
dieselben Werte wie auf den Folien. Ändern Sie den Seed, und Sie bekommen neue Würfe.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PETROL, ROST = "#4F8F88", "#B7684E"

## 1 · 100 000 Würfe (Folie 6)

Wo landet der Anteil der Sechsen, wenn wir öfter würfeln?

In [ ]:
rng = np.random.default_rng(2026)
wuerfe = rng.integers(1, 7, size=100_000)          # 100 000 Würfe
anteil = (wuerfe == 6).cumsum() / np.arange(1, 100_001)
for n in [10, 100, 1_000, 100_000]:
    print(f"nach {n:>6} Würfen: {anteil[n - 1]:.1%} Sechsen")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, 100_001), anteil, color=PETROL)
plt.axhline(1 / 6, color=ROST, ls="--", label="1/6")
plt.xscale("log")
plt.xlabel("Anzahl Würfe")
plt.ylabel("Anteil Sechsen")
plt.legend()
plt.show()

## 2 · Geburtstage (Folie 10)

Regel 1: unabhängige Wahrscheinlichkeiten multiplizieren. Regel 2: „mindestens zwei gleich“ = 1 − „alle verschieden“.

In [ ]:
def mindestens_zwei_gleich(k):
    alle_verschieden = 1.0
    for i in range(k):
        alle_verschieden *= (365 - i) / 365   # die nächste Person braucht einen noch freien Tag
    return 1 - alle_verschieden


for k in [3, 9, 23, 41, 57, 75]:
    print(f"{k:>2} Personen: {mindestens_zwei_gleich(k):.2%}")

personen = range(1, 71)
plt.figure(figsize=(8, 4))
plt.plot(personen, [mindestens_zwei_gleich(k) for k in personen], color=PETROL)
plt.xlabel("Personen im Raum")
plt.ylabel("mindestens zwei gleich")
plt.show()

## 3 · Die Bank an einem Abend (Folie 14)

Eine Million Runden zu je 1 € auf Rot. Erwartet: 1 000 000 · 1/37 ≈ 27 027 € für die Bank.

In [ ]:
rng = np.random.default_rng(37)
rot_kommt = rng.integers(0, 37, size=1_000_000) < 18   # 18 der 37 Felder sind rot
bank = np.where(rot_kommt, -1, 1)                      # Sicht der Bank: Rot kostet sie 1 €, sonst gewinnt sie 1 €
print("Gewinn der Bank:", bank.sum(), "€")
je_euro = bank.cumsum() / np.arange(1, 1_000_001)
for n in [10, 100, 1_000, 1_000_000]:
    print(f"nach {n:>7} Runden: {je_euro[n - 1]:+.1%} je Euro")

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, 1_000_001), je_euro, color=PETROL)
plt.axhline(1 / 37, color=ROST, ls="--", label="+2,7 % für die Bank")
plt.axhline(0, color="grey", lw=0.8)
plt.xscale("log")
plt.ylim(-0.6, 0.6)
plt.xlabel("gespielte Runden")
plt.ylabel("Gewinn der Bank je 1 €")
plt.legend()
plt.show()

## 4 · 1 000 Abende: Rot oder eine Zahl? (Folie 16)

Jeder Abend: 100 Runden zu je 1 €. Rot zahlt 1 : 1, eine einzelne Zahl (hier die 17) zahlt 35 : 1.

In [ ]:
rng = np.random.default_rng(100)
rot   = np.where(rng.integers(0, 37, (1000, 100)) < 18, 1, -1).sum(axis=1)
plein = np.where(rng.integers(0, 37, (1000, 100)) == 17, 35, -1).sum(axis=1)
for name, x in [("Rot", rot), ("eine Zahl", plein)]:
    print(f"{name:>9}: Mittel {x.mean():6.2f} € · Standardabweichung {x.std():4.1f} € · von {x.min()} bis {x.max()} €")
print(f"eine Zahl: {(plein == -100).mean():.1%} der Abende alles weg, {(plein > 0).mean():.0%} mit Gewinn")

fig, achsen = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
klassen = np.arange(-104, 229, 8)
achsen[0].hist(rot, bins=klassen, color=PETROL)
achsen[0].set_title("auf Rot")
achsen[1].hist(plein, bins=klassen, color=ROST)
achsen[1].set_title("auf eine Zahl")
for ax in achsen:
    ax.set_xlabel("Ergebnis nach 100 Runden (€)")
achsen[0].set_ylabel("Abende")
plt.show()

## 5 · Standardabweichung in vier Schritten (Folie 17)

Abweichung vom Mittelwert, quadrieren, Durchschnitt (= Varianz), Wurzel: eine Zeile pro Schritt. Im Praktikum rechnen
Sie dasselbe für alle Olist-Lieferungen.

In [ ]:
verspaetung = pd.DataFrame({"Flixbus": [10, 15, 20, 15, 15], "ICE": [0, 0, 5, 10, 60]})   # Minuten an fünf Tagen
mittel = verspaetung.mean()
abweichung = verspaetung - mittel          # Schritt 1
quadrate = abweichung ** 2                 # Schritt 2
varianz = quadrate.mean()                  # Schritt 3
standardabweichung = np.sqrt(varianz)      # Schritt 4
print(pd.DataFrame({"Mittel": mittel, "Varianz": varianz, "Standardabweichung": standardabweichung}).round(1))

## 6 · Zum Nachrechnen

Alle weiteren Zahlen der Folien. Rechnen Sie zuerst von Hand, dann hier.

Der echte ICE 1011 (Dortmund – Düsseldorf – München), Verspätung in Minuten vom 28.09. bis 02.10.2026.
Hinweis: pandas `.std()` teilt durch n − 1 statt durch n; deshalb hier `np.std`, wie in der Vorlesung.

In [ ]:
ice_1011 = np.array([111, 8, 25, 216, 6])
print(f"Mittel {ice_1011.mean():.1f} · Median {np.median(ice_1011):.0f} · Standardabweichung {ice_1011.std():.1f} Minuten")

**Die Kneipe (Folien 20 und 21):** Mittelwert, Median, Quartile, Ausreißer, dann kommt Elon Musk herein.

In [ ]:
vermoegen = pd.Series([2_000, 8_000, 15_000, 25_000, 40_000, 60_000, 90_000, 150_000, 310_000])
q1, q3 = vermoegen.quantile(0.25), vermoegen.quantile(0.75)
iqr = q3 - q1
print(f"Mittelwert {vermoegen.mean():.0f} € · Median {vermoegen.median():.0f} €")
print(f"Q1 {q1:.0f} € · Q3 {q3:.0f} € · IQR {iqr:.0f} € · Ausreißer über {q3 + 1.5 * iqr:.0f} €")

mit_musk = pd.concat([vermoegen, pd.Series([8e11])])   # Elon Musk, rund 800 Mrd. € (Forbes)
print(f"mit Musk: Mittelwert {mit_musk.mean() / 1e9:.0f} Mrd. € · Median {mit_musk.median():.0f} €")

vermoegen.plot.box(figsize=(4, 4), ylabel="Vermögen (€)")
plt.show()

**Hans spielt gern Klavier (Folie 23):** natürliche Häufigkeiten statt Prozent.

In [ ]:
ingenieure, musiker = 6_700, 100           # etwa 67 Ingenieure je Musiker (Bundesagentur für Arbeit)
klavier_ing = ingenieure * 0.05            # 5 von 100 Ingenieuren spielen gern Klavier
klavier_mus = musiker                      # großzügig: alle Musiker
print(f"P(Ingenieur | spielt gern Klavier) = {klavier_ing:.0f} / {klavier_ing + klavier_mus:.0f} "
      f"= {klavier_ing / (klavier_ing + klavier_mus):.0%}")

**Zwei Klassen (Folie 24):** Eine Person wechselt, beide Durchschnitte steigen.

In [ ]:
a, b = [60, 70], [75, 90, 95]
print(f"vorher:  A {np.mean(a):.1f} · B {np.mean(b):.1f} · alle {np.mean(a + b):.1f}")
a, b = a + [75], [90, 95]                  # die Person mit 75 Punkten wechselt von B nach A
print(f"nachher: A {np.mean(a):.1f} · B {np.mean(b):.1f} · alle {np.mean(a + b):.1f}")

**Berkeley 1973 (Folie 25):** erst je Fachbereich, dann zusammen.

In [ ]:
berkeley = pd.DataFrame({"Fachbereich": ["A", "A", "F", "F"], "Gruppe": ["Männer", "Frauen", "Männer", "Frauen"],
                         "Bewerbungen": [825, 108, 373, 341], "zugelassen": [512, 89, 22, 24]})
berkeley["Quote"] = berkeley["zugelassen"] / berkeley["Bewerbungen"]
print(berkeley.round(2), "\n")
zusammen = berkeley.groupby("Gruppe")[["Bewerbungen", "zugelassen"]].sum()
zusammen["Quote"] = zusammen["zugelassen"] / zusammen["Bewerbungen"]
print(zusammen.round(3))

**Zum Weiterdenken:** Ändern Sie in Teil 1 den Seed oder die Zahl der Würfe. Ab wie vielen Würfen liegt der Anteil
verlässlich zwischen 16 % und 17,5 %? Das ist die Frage von Woche 3.